# Variant C on Kaggle (classical-CV lesion fusion)


In [ ]:
!nvidia-smi


In [ ]:
!git clone https://github.com/Dilshan-Fernando-01/Computer-Vision-Assignment.git
%cd Computer-Vision-Assignment


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import glob
import os


matches = glob.glob("/kaggle/input/**/DR_grading.csv", recursive=True)
assert matches, "Dataset not attached - add mariaherrerot/ddrdataset via Add Input first"

os.environ["DDR_GRADING_CSV"] = matches[0]
os.environ["DDR_IMAGES_DIR"] = os.path.join(os.path.dirname(matches[0]), "DR_grading", "DR_grading")
print("Dataset found at", os.environ["DDR_IMAGES_DIR"])


In [ ]:
!ls data/processed/splits/


In [ ]:
import sys
sys.path.insert(0, '.')

import json
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, WeightedRandomSampler

from src.datasets.dataset import DRGradingDataset
from src.augmentation.augment import get_training_augmentations, make_sample_weights
from src.models.fusion import build_fusion_model
from src.training.train import fit, evaluate, get_device

DEVICE = get_device()
IMAGE_SIZE = 512
BATCH_SIZE = 32
EPOCHS = 40
PATIENCE = 8
LR = 1e-4

print('device:', DEVICE)

train_ds = DRGradingDataset('data/processed/splits/train.csv', image_size=IMAGE_SIZE,
                             transform=get_training_augmentations(IMAGE_SIZE, with_lesion=True),
                             with_lesion_map=True)
val_ds   = DRGradingDataset('data/processed/splits/val.csv',   image_size=IMAGE_SIZE, with_lesion_map=True)
test_ds  = DRGradingDataset('data/processed/splits/test.csv',  image_size=IMAGE_SIZE, with_lesion_map=True)

train_labels = [label for _, label in train_ds.samples]
sample_weights = make_sample_weights(train_labels)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, num_workers=4)

print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))


In [ ]:
criterion_c = torch.nn.CrossEntropyLoss()
model_c = build_fusion_model('efficientnet_b0', num_classes=5, pretrained=True)

os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/history', exist_ok=True)

history_c = fit(
    model_c, train_loader, val_loader, criterion_c,
    epochs=EPOCHS, lr=LR, patience=PATIENCE,
    checkpoint_path='outputs/checkpoints/variant_c_fusion_efficientnet_b0.pt',
    device=DEVICE,
)

with open('outputs/history/variant_c_history.json', 'w') as f:
    json.dump(history_c, f, indent=2)
print('best val macro F1:', history_c['best_val_macro_f1'])


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_c['train_loss'], label='train'); axes[0].plot(history_c['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history_c['train_acc'], label='train'); axes[1].plot(history_c['val_acc'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend()
axes[2].plot(history_c['val_macro_f1'], color='green'); axes[2].set_title('Val Macro F1')
fig.suptitle('Variant C (lesion fusion) - DDR (Kaggle)')
fig.tight_layout()
fig.savefig('outputs/history/variant_c_curves.png', dpi=120)
plt.show()


In [ ]:
model_c.load_state_dict(torch.load('outputs/checkpoints/variant_c_fusion_efficientnet_b0.pt', map_location=DEVICE))
test_metrics_c = evaluate(model_c, test_loader, criterion_c, DEVICE)

report = classification_report(test_metrics_c['labels'], test_metrics_c['preds'], target_names=[f'Stage {i}' for i in range(5)])
cm = confusion_matrix(test_metrics_c['labels'], test_metrics_c['preds'])

print('Test accuracy:', test_metrics_c['accuracy'])
print('Test macro F1:', test_metrics_c['macro_f1'])
print(report)
print(np.array(cm))

with open('outputs/history/variant_c_test_report.txt', 'w') as f:
    f.write(f"Test accuracy: {test_metrics_c['accuracy']}\nTest macro F1: {test_metrics_c['macro_f1']}\n\n")
    f.write(report)
    f.write('\nConfusion matrix:\n')
    f.write(np.array2string(cm))
